In [1]:
import json
from langchain_community.llms import Ollama

In [3]:
# 1. Inisialisasi Model
llm_base = Ollama(model="llama3.2:1b-instruct-q4_K_M", temperature=0.3)

def generate_no_rag(question, model):
    prompt = f"""Pertanyaan: {question}
Jawaban akurat, lengkap dan langsung (to the poin) berdasarkan pengetahuan hukum daerah Indonesia:"""
    # Menggunakan invoke sesuai versi terbaru LangChain
    return model.invoke(prompt).strip()

input_file = '/Users/novay/Applications/Project/Enterwind/lexindo/python/lexindo-llm/05_evaluation_set/ragas_eval_beautified.jsonl'
output_file = '/Users/novay/Applications/Project/Enterwind/lexindo/python/lexindo-llm/05_evaluation_set/ragas_eval.jsonl'

In [4]:
# 2. Proses Looping
updated_data = []

# Kita buka file yang formatnya "beautified" (banyak baris per objek)
# Maka kita perlu memuatnya sebagai list of dicts jika file tersebut valid JSON
# Namun, jika file kamu adalah kumpulan JSON terpisah, gunakan logika ini:

with open(input_file, 'r', encoding='utf-8') as f:
    content = f.read()
    # Mencari pola objek JSON { ... } untuk menangani format beautified
    import re
    blocks = re.findall(r'\{.*?\}(?=\n\{|\n$|$)', content, re.DOTALL)
    
    print(f"Memulai proses {len(blocks)} data...")

    for i, block in enumerate(blocks):
        data = json.loads(block)
        question = data['question']
        
        print(f"[{i+1}/{len(blocks)}] Generating answer for: {question[:50]}...")
        
        # Eksekusi LLM
        try:
            answer = generate_no_rag(question, llm_base)
            # Masukkan hasil ke kolom answer_base
            data['answer_base'] = answer
        except Exception as e:
            print(f"Error pada baris {i+1}: {e}")
            data['answer_base'] = "ERROR_GENERATION"

        updated_data.append(data)

Memulai proses 1119 data...
[1/1119] Generating answer for: Apa tujuan Manajemen Sumber Daya Manusia menurut P...
[2/1119] Generating answer for: Di mana Penjabaran Laporan Realisasi Anggaran dapa...
[3/1119] Generating answer for: Apa isi Pasal 7 mengenai Belanja Daerah?...
[4/1119] Generating answer for: Bagaimana kedudukan Dinas Komunikasi dan Informati...
[5/1119] Generating answer for: Apa definisi dari Lembaga Kerjasama Tripartit (LKS...
[6/1119] Generating answer for: Sebutkan rincian rencana Belanja Operasi sesuai Pa...
[7/1119] Generating answer for: Apa isi pembinaan dan pengawasan menurut Pasal 32?...
[8/1119] Generating answer for: Apa saja syarat khusus bagi aparatur kebencanaan u...
[9/1119] Generating answer for: Apa saja persyaratan pendaftaran Penerima beasiswa...
[10/1119] Generating answer for: Sebutkan daftar peraturan menteri yang menjadi das...
[11/1119] Generating answer for: Siapa yang memimpin Koordinator Wilayah Kecamatan ...
[12/1119] Generating answer for: P

KeyboardInterrupt: 

In [ ]:
# 3. Simpan kembali ke format MINIFIED (satu baris satu JSON)
with open(output_file, 'w', encoding='utf-8') as f:
    for entry in updated_data:
        # json.dumps otomatis mengubah newline menjadi \n
        json_line = json.dumps(entry, ensure_ascii=False)
        f.write(json_line + '\n')

print(f"\nSelesai! Hasil akhir disimpan di: {output_file}")